In [ ]:
#Model 1: 30 neuron network
#Import all packages
import numpy as np
import matplotlib as plt
import matplotlib.pyplot as plt
import networkx as nx
from scipy.linalg import expm
from matplotlib.colors import hsv_to_rgb
from scipy.optimize import linprog
#First, flat neuron
def build_graph_cell1(rows=5, cols=6):
    G = nx.grid_2d_graph(rows,cols, periodic=True)
    mapping = {(r, c): r * cols + c for r, c in G.nodes()}
    return nx.relabel_nodes(G, mapping, copy=True)

#Second, hyperbolic network:
def build_graph_cell2():
    adj = {
        0:  [11,12,18,19],
        1:  [2, 10, 23, 14],
        2:  [1, 3, 13, 14],
        3:  [2, 4, 13, 17],
        4:  [3, 5, 16, 17],
        5:  [4, 6, 16, 20],
        6:  [5, 7, 19, 20],
        7:  [6, 8, 19, 22],
        8:  [9, 7, 21, 22],
        9:  [8, 10, 11, 21],
        10: [1, 9, 11, 23],
        11: [10, 12, 9, 0],
        12: [11, 13, 28, 0],
        13: [2, 3, 12, 28],
        14: [1, 2, 29, 15],
        15: [14, 16, 24, 29],
        16: [4, 5, 15, 24],
        17: [3, 4, 18, 26],
        18: [17, 19, 26, 0],
        19: [6, 7, 18, 0],
        20: [5, 6, 27, 28],
        21: [8, 9, 27, 29],
        22: [7, 8, 24, 25],
        23: [1, 10, 25, 26],
        24: [15, 16, 22, 25],
        25: [22, 23, 24, 26],
        26: [17, 18, 23, 25],
        27: [20, 21, 28, 29],
        28: [12, 13, 20, 27],
        29: [14, 15, 21, 27],
    }
    G = nx.Graph()
    for u, nbrs in adj.items():
        for v in nbrs:
            G.add_edge(u,v)
    return G

#Compute Ollivier-Ricci curvature
def ollivier_ricci_curvature(G, u, v, alpha=0.0):
    Nu = list(G.neighbors(u))
    Nv = list(G.neighbors(v))
    supp_u = [u] + Nu
    supp_v = [v] + Nv
    all_nodes = list(set(supp_u + supp_v))
    node_idx = {node: i for i, node in enumerate(all_nodes)}
    m = len(all_nodes)
    mu = np.zeros(m)
    mv = np.zeros(m)
    mu[node_idx[u]] = alpha
    for nb in Nu:
        mu[node_idx[nb]] += (1 - alpha) / len(Nu)
    
    mv[node_idx[v]] = alpha
    for nb in Nv:
        mv[node_idx[nb]] += (1 - alpha) / len(Nv)
    D = np.zeros((m,m))
    for i, ni in enumerate(all_nodes):
        for j, nj in enumerate(all_nodes):
            if i <= j:
                d = nx.shortest_path_length(G, ni, nj)
                D[i, j] = d
                D[j, i] = d 
    c = D.flatten()
    A_eq = []
    b_eq = []
    for i in range(m):
        row = np.zeros(m * m)
        row[i * m: (i+1) *m] = 1.0
        A_eq.append(row)
        b_eq.append(mu[i])
    for j in range(m):
        col = np.zeros(m * m)
        col[j::m] = 1.0
        A_eq.append(col)
        b_eq.append(mv[j])
    A_eq = np.array(A_eq)
    b_eq = np.array(b_eq)
    bounds = [(0, None)] * (m * m)
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")
    W1 = res.fun
    d_uv = nx.shortest_path_length(G, u, v)
    return 1.0 - W1 / d_uv

#Create G1: flat and G2: Hyperbolic
G1 = build_graph_cell1(rows=5, cols=6)
G2 = build_graph_cell2()

#visualize Ollivier-Ricci curvature
G1_edge_curvatures = {}
for u, v in G1.edges():
    kappa = ollivier_ricci_curvature(G1, u, v, alpha=0.0)
    G1_edge_curvatures[(u, v)] = kappa
G1_curvature_vals = np.array(list(G1_edge_curvatures.values()))

G2_edge_curvatures = {}
for u, v in G2.edges():
    kappa = ollivier_ricci_curvature(G2, u, v, alpha=0.0)
    G2_edge_curvatures[(u, v)] = kappa
G2_curvature_vals = np.array(list(G2_edge_curvatures.values()))


# --- Side-by-side visualization of edge Ollivier-Ricci curvature for G1 and G2 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# Use a shared color scale across both graphs
all_curv = np.concatenate([G1_curvature_vals, G2_curvature_vals])
vmin, vmax = float(all_curv.min()), float(all_curv.max())
pos1 = nx.spring_layout(G1, seed=0)
pos2 = nx.spring_layout(G2, seed=0)

# ---- G1 ----
edges1 = list(G1_edge_curvatures.keys())
colors1 = [G1_edge_curvatures[e] for e in edges1]

nx.draw_networkx_nodes(
    G1, pos1, ax=axes[0],
    node_size=80, node_color="lightgray",
    edgecolors="black", linewidths=0.5
)
ec1 = nx.draw_networkx_edges(
    G1, pos1, ax=axes[0],
    edgelist=edges1, edge_color=colors1,
    edge_cmap=plt.cm.RdBu, width=2.0, alpha=0.9,
    edge_vmin=vmin, edge_vmax=vmax
)
axes[0].set_title(f"G1 (flat): mean κ = {np.mean(G1_curvature_vals):.4f}")
axes[0].axis("off")

# ---- G2 ----
edges2 = list(G2_edge_curvatures.keys())
colors2 = [G2_edge_curvatures[e] for e in edges2]
nx.draw_networkx_nodes(
    G2, pos2, ax=axes[1],
    node_size=80, node_color="lightgray",
    edgecolors="black", linewidths=0.5
)
ec2 = nx.draw_networkx_edges(
    G2, pos2, ax=axes[1],
    edgelist=edges2, edge_color=colors2,
    edge_cmap=plt.cm.RdBu, width=2.0, alpha=0.9,
    edge_vmin=vmin, edge_vmax=vmax
)
axes[1].set_title(f"G2 (hyperbolic): mean κ = {np.mean(G2_curvature_vals):.4f}")
axes[1].axis("off")

cbar = fig.colorbar(ec2, ax=axes[1], fraction=0.046, pad=0.04)
cbar.set_label("Ollivier-Ricci curvature κ")
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.pylab import zeros

# Ensure graphs exist
if "G1" not in globals() or "G2" not in globals():
    raise NameError("Run Cell 1 first so G1 and G2 are defined.")

# Laplacians and initial states
L1 = nx.laplacian_matrix(G1).astype(float).toarray()
L2 = nx.laplacian_matrix(G2).astype(float).toarray()

rng = np.random.default_rng(0)
Z1 = 0.3 * (rng.standard_normal(L1.shape[0]) + 1j * rng.standard_normal(L1.shape[0]))

Z1 = np.zeros(L1.shape[0], dtype=np.complex128)
Z2 = 0.3 * (rng.standard_normal(L2.shape[0]) + 1j * rng.standard_normal(L2.shape[0]))
Z2 = np.zeros(L2.shape[0], dtype=np.complex128)
# Small constant drive so dynamics do not collapse to zero
I1 = np.zeros(L1.shape[0], dtype=np.complex128)
I2 = np.zeros(L2.shape[0], dtype=np.complex128)
I1[0] = 0.01j
I2[0] = 0.01j
# Fixed layouts for movie stability
pos1 = nx.spring_layout(G1, seed=0)
pos2 = nx.spring_layout(G2, seed=0)

# Precompute spectral decomposition once (faster for animation)
lam1, Q1 = np.linalg.eigh(L1)
lam2, Q2 = np.linalg.eigh(L2)


def step_precomp(Z, lam, Q, I):
    Z_hat = Q.T @ Z
    Y_hat = np.exp(1j * lam) * Z_hat
    Y = Q @ Y_hat
    Y = Y + I
    Z_next = Y / np.sqrt(1.0 + np.abs(Y)**2)
    return Z_next


def state_to_rgb(z):
    mag = np.abs(z)
    scale = np.percentile(mag, 99) if np.any(mag) else 1.0
    scale = float(scale) if scale > 0 else 1.0
    v = np.clip(mag / scale, 0.0, 1.0)
    h = (np.angle(z) + np.pi) / (2 * np.pi)
    s = np.ones_like(h)
    rgb = hsv_to_rgb(np.stack([h, s, v], axis=-1))
    return rgb, scale


T = 100
states1 = [Z1.copy()]
states2 = [Z2.copy()]
for _ in range(T):
    Z1 = step_precomp(Z1, lam1, Q1, I1)
    Z2 = step_precomp(Z2, lam2, Q2, I2)
    states1.append(Z1.copy())
    states2.append(Z2.copy())

fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)


def draw_panel(ax, G, pos, z, title_prefix):
    rgb, scale = state_to_rgb(z)
    ax.clear()
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25, width=1.0)
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=rgb,
        node_size=260,
        linewidths=1.0,
        edgecolors="black"
    )
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
    ax.set_title(f"{title_prefix} | hue=phase, brightness=|z| (p99={scale:.3g})")
    ax.axis("off")


def update(t):
    draw_panel(axes[0], G1, pos1, states1[t], f"G1 at t={t}")
    draw_panel(axes[1], G2, pos2, states2[t], f"G2 at t={t}")
    return []

ani = FuncAnimation(fig, update, frames=T + 1, interval=120, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def phi(z):
    return z / np.sqrt(1.0 + np.abs(z)**2)

def simulate_with_unitary_laplacian(G, T=100, dt=1.0, I=0.01j):
    n = G.number_of_nodes()

    # Graph Laplacian L and anti-Hermitian generator A = i*dt*L
    A_adj = nx.to_numpy_array(G, nodelist=range(n), dtype=float)
    deg = A_adj.sum(axis=1)
    L = np.diag(deg) - A_adj
    A = 1j * dt * L

    # Unitary kernel
    U = expm(A)

    # Sanity check
    unitarity_err = np.max(np.abs(U.conj().T @ U - np.eye(n)))

    # Recurrent update z <- phi(U z + I_vec)
    z = np.zeros(n, dtype=np.complex128)
    I_vec = np.zeros(n, dtype=np.complex128)
    I_vec[0] = I

    states = [z.copy()]
    for _ in range(T):
        z_lin = U @ z + I_vec
        z = phi(z_lin)
        states.append(z.copy())

    return states, unitarity_err

def state_to_rgb(z):
    # phase as hue, magnitude as brightness
    mag = np.abs(z)
    scale = np.percentile(mag, 99) if np.any(mag) else 1.0
    scale = float(scale) if scale > 0 else 1.0
    v = np.clip(mag / scale, 0.0, 1.0)
    h = (np.angle(z) + np.pi) / (2 * np.pi)
    s = np.ones_like(h)
    rgb = hsv_to_rgb(np.stack([h, s, v], axis=-1))
    return rgb, scale

if "G1" not in globals() or "G2" not in globals():
    raise NameError("Run Cell 1 first so G1 and G2 are defined.")

T = 100
dt = 1.0
I = 0.01j

states1, err1 = simulate_with_unitary_laplacian(G1, T=T, dt=dt, I=I)
states2, err2 = simulate_with_unitary_laplacian(G2, T=T, dt=dt, I=I)

print(f"G1 max |U†U - I| = {err1:.3e}")
print(f"G2 max |U†U - I| = {err2:.3e}")

pos1 = nx.spring_layout(G1, seed=0)
pos2 = nx.spring_layout(G2, seed=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

def draw_panel(ax, G, pos, z, title_prefix):
    rgb, scale = state_to_rgb(z)
    ax.clear()
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.25, width=1.0)
    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=rgb,
        node_size=260,
        linewidths=1.0,
        edgecolors="black"
    )
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
    ax.set_title(f"{title_prefix}\nhue=phase, brightness=|z| (p99={scale:.3g}), I={I}")
    ax.axis("off")

def update(t):
    draw_panel(axes[0], G1, pos1, states1[t], f"G1 at t={t}")
    draw_panel(axes[1], G2, pos2, states2[t], f"G2 at t={t}")
    return []

ani = FuncAnimation(fig, update, frames=T + 1, interval=120, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

In [ ]:
def compute_L_and_U(graph):
    n = graph.number_of_nodes()
    A = nx.to_numpy_array(graph, nodelist=range(n), dtype=float)
    D = np.diag(A.sum(axis=1))
    L_graph = 1j * (D - A)

    M = np.random.randn(n, n) + 1j * np.random.randn(n, n)
    L_rand = (M - M.conj().T) / 2
    U_unitary = expm(L_rand)
    return L_graph, L_rand, U_unitary

L1_graph, L1_rand, U1 = compute_L_and_U(G1)
L2_graph, L2_rand, U2 = compute_L_and_U(G2)

fig, ax = plt.subplots(1, 2, figsize=(10, 8), constrained_layout=True)
ax[0].imshow(np.imag(L1_graph)); ax[0].set_title("Im(i*Laplacian) for Euclidean")
ax[1].imshow(np.imag(L2_graph)); ax[1].set_title("Im(i*Laplacian) for Hyperbolic")
for a in ax.ravel():
    a.set_xticks([])
    a.set_yticks([])
plt.show()
